In [1]:
from bs4 import BeautifulSoup
import requests

In [11]:
def extract_text_from_pages(start_url):

    i = 0
    page_list=[]
    while i < 40:
        response = requests.get(start_url)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser", from_encoding=response.encoding)
            
            page_text = soup.find("div", class_="mwsgeneric-base-html parbase section")

            if page_text:
                for panel_body in page_text.find_all('div', class_='panel-body'):
                    panel_body.extract()

            page_list.append(page_text.text)
        
            next_button = soup.find("a", href=True, rel="next")
            if next_button:
                start_url = "https://www.canada.ca" + next_button['href']
                i += 1
            else:
                break
        else:
            print("Failed to fetch", start_url)
            break
    return page_list 

In [20]:
start_url = "https://www.canada.ca/fr/emploi-developpement-social/programmes/assurance-emploi/ae-liste/rapports/guide/ch-25/pouvoir-legislatif.html#h2.01"
full_text = extract_text_from_pages(start_url)

In [4]:
import re
import json

In [16]:
def split_text(input_text):
    # Split the input text into sections using regex

    sections = re.split(r'\n(\d+\.\d+\.\d+\.?\d?)\s', input_text.strip())

    # Define a list to store formatted section data

    formatted_sections = [] 

    # Iterate over sections to extract information
    for i in range(1, len(sections), 2):
        section_number = sections[i]
        section_content = sections[i + 1].split('\n', 1)
        section_title = section_content[0].strip()
        section_text = section_content[1].strip() if len(section_content) > 1 else ''

        # Check if the section text contains the footnote marker
        # if '[' in section_text:
            # section_text = section_text.split('[')[0].strip()

        # Append formatted section data to the list
        formatted_sections.append({
            "section_number": section_number,
            "section_title": section_title,
            "section_text": section_text
        })

    # Print the formatted sections as JSON
    return json.dumps(formatted_sections, indent=2, ensure_ascii=False)

In [21]:
with open('output_french_full.txt', 'a', encoding='utf-8') as file:
    for i in full_text:
        result = split_text(i)
        file.write(result + '\n')

In [22]:
def text_clean(text_file):
    with open(text_file, 'r') as file:
        text = file.read()
        # mod_text = text.replace('\u2019', "'").replace('\u2013', "-").replace('u\201c', '"').replace('u\201d', '"').replace('\u2018', "'")
        # mod_text1 = mod_text.replace('\u00bd', '1/2').replace('\u00a0', ' ').replace('\u00e9', 'é').replace('\u00ea', 'ê').replace('\u00e8', 'è')
        # pattern = r'Footnote \d\d?'
        # mod_text = re.sub(pattern, '', text)
        # pattern1 =  r'\\?\"'
        # mod_text = re.sub(pattern1,'', text)

    with open('test_output_french_clean.txt', 'w') as clean_file:
        clean_file.write(text)

In [18]:
text_clean('output_french_test.txt')

In [23]:
import unicodedata

def normalize_unicode(text):
    # Normalize Unicode characters in a string to their closest ASCII representation.
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')

# Read the original content of the file
with open('output_french_test.txt', 'r', encoding='utf-8') as file:
    content = file.read()

# Normalize the content
normalized_content = normalize_unicode(content)

# Write the normalized content to a new file
with open('output_french_clean.txt', 'w', encoding='ascii') as file:
    file.write(normalized_content)